In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["MKL_CBWR"] = "COMPATIBLE"
os.environ["ATEN_CPU_CAPABILITY"] = "avx2"
os.environ["OMP_NUM_THREADS"] = os.environ["MKL_NUM_THREADS"] = os.environ["OPENBLAS_NUM_THREADS"] = "8"
os.environ["NPY_DISABLE_CPU_FEATURES"] = "AVX512F AVX512CD AVX512_KNL AVX512_KNM AVX512_SKX AVX512_CLX AVX512_CNL AVX512_ICL AVX512_SPR"
os.environ["OPENBLAS_CORETYPE"] = "Haswell"
# JAX (GW solves) and PyTorch (φ/ψ training) share the GPU. Stop JAX from reserving
# most of the card upfront; it allocates as needed, up to 95% for the final GW solve.
# Must be set before JAX is imported.
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.95"

import sys
sys.path.append("..")

# Human–Mouse Alignment (HESTA → MOSTA)

We align a sagittal section of a human embryo from HESTA (https://db.cngb.org/hesta/download/) to a sagittal section of a mouse embryo from MOSTA (Chen et al. '22) with MGW. The human section is the source A and the mouse section the target B, so the coupling `P` is HESTA × MOSTA and carries mouse labels and expression onto the human section.

**Which sections.** Neither atlas records where a section lies along the left–right axis. The laterality run (`laterality_test/laterality_score.ipynb`, results in `laterality/`) scores each section from 0 (midline) to 1 (far lateral) and pairs every HESTA section with the MOSTA section of the matched stage whose score is closest (`best`). **Pick the HESTA section in the next cell**; its MOSTA partner is read from `laterality/laterality_pairs.csv` unless `MOSTA_SECTION` is set.

**Control.** Cross-species differences only mean something relative to the differences between two individuals of the same species. The notebook therefore also aligns the HESTA section with a section of *another* HESTA embryo of the same stage (closest laterality score) and reports both runs side by side.

**Features.** `FEATURE_MODE` selects what MGW sees:
- `"native"`: each species keeps its own genes and PCA. The CCA feeler learns the map between the two feature spaces, so no ortholog knowledge enters the alignment ("multimodal", as in `mouse_align.ipynb`).
- `"orthologs"`: both sections are restricted to one-to-one orthologs (Ensembl BioMart, `mgw/orthologs.py`), named by the human symbol, with a joint PCA as in `hesta_align.ipynb`.

In both modes the orthologs are used to evaluate the coupling (co-expression check below).

In [ ]:
# ============================================================
# Choose the sections to align
# ============================================================
HESTA_SECTION   = "CS12-13_E2S6"  # source A (human); must appear in laterality_pairs.csv
MOSTA_SECTION   = None            # target B (mouse); None: the 'best' partner from laterality_pairs.csv
CONTROL_SECTION = None            # HESTA control; None: other embryo, same stage, closest laterality score
RUN_CONTROL     = True

FEATURE_MODE = "native"           # "native" (per-species genes + PCA) or "orthologs" (1:1 orthologs + joint PCA)

# Folders (None: <MGW data dir>/hesta, <MGW data dir>/mosta or mouse_embryo, <repo>/laterality,
# <MGW data dir>/orthologs/human_mouse_one2one.tsv)
HESTA_DIR      = None
MOSTA_DIR      = None
LATERALITY_DIR = None
ORTHOLOG_TSV   = None

# Bins kept per section (random subsample). The GW solve keeps several n x n matrices on the
# GPU; 14k fits a 16 GB card (see hesta_align.ipynb). None keeps every bin.
MAX_OBS = 14_000
# Labels removed before subsampling. Cavity is ~40% of some HESTA sections but a few % of
# MOSTA sections, which would force mass onto the wrong partners in a balanced coupling.
DROP_LABELS = ["Cavity", "Extraembryonic Tissue"]

# MGW hyperparameters (as in hesta_align.ipynb)
PCA_COMP     = 30
CCA_COMP     = 3
PHI_ARC      = (128, 256, 256, 128)
KNN_K        = 12
GEODESIC_EPS = 1e-1
LR           = 1e-3
N_ITER       = 20_000
GW_PARAMS    = dict(verbose=True, inner_maxit=3000, outer_maxit=3000, inner_tol=1e-7, outer_tol=1e-7, epsilon=1e-4)
SEED         = 1
PLOT_NET     = False   # plot φ/ψ fits of a few features

N_EVAL_GENES = 500     # most variable 1:1 orthologs used for the co-expression check

In [ ]:
import gc
from datetime import datetime
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import torch
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from scipy.spatial import cKDTree
from scipy.stats import rankdata

import mgw as mgw_pkg
from mgw import align_vis, orthologs, plotting, util
from mgw import mgw
from mgw.config import get_data_dir

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_default_dtype(torch.float64)
print("Device:", device)

REPO = Path(mgw_pkg.__file__).resolve().parent.parent
DATA = get_data_dir()  # asks on first run, then remembered per machine
HESTA_DIR = Path(HESTA_DIR or DATA / "hesta").expanduser()
if MOSTA_DIR is None:
    MOSTA_DIR = next((DATA / d for d in ["mosta", "mouse_embryo"] if (DATA / d).is_dir()), DATA / "mosta")
MOSTA_DIR = Path(MOSTA_DIR).expanduser()
LATERALITY_DIR = Path(LATERALITY_DIR or REPO / "laterality").expanduser()
for name, d in [("HESTA", HESTA_DIR), ("MOSTA", MOSTA_DIR), ("laterality", LATERALITY_DIR)]:
    print(f"{name} directory: {d}")

ORTHO = orthologs.load_one2one(ORTHOLOG_TSV)  # builds the table from BioMart on first use
print(f"one-to-one orthologs: {len(ORTHO):,}")

SAVE_DIR = HESTA_DIR / "alignments"
SAVE_DIR.mkdir(parents=True, exist_ok=True)
SUMMARY_CSV = SAVE_DIR / "human_mouse_summary.csv"

DOWNLOAD_URL = {
    "HESTA": "https://ftp.cngb.org/pub/SciRAID/stomics/STDS0000394/stomics/{}_HESTA.h5ad",
    "MOSTA": "https://ftp.cngb.org/pub/SciRAID/stomics/STDS0000058/stomics/{}.MOSTA.h5ad",
}

def section_path(dataset, sec):
    return HESTA_DIR / f"{sec}_HESTA.h5ad" if dataset == "HESTA" else MOSTA_DIR / f"{sec}.MOSTA.h5ad"

def check_files(*specs):
    missing = [(ds, s) for ds, s in specs if not section_path(ds, s).exists()]
    if missing:
        cmds = "\n".join(f"curl -L -C - -o {section_path(ds, s)} {DOWNLOAD_URL[ds].format(s)}" for ds, s in missing)
        raise FileNotFoundError(f"missing sections; download them with\n{cmds}")

## Pair selection from the laterality run

`laterality_pairs.csv` lists, for every HESTA section, the MOSTA sections of the matched stage(s) (notes, Table 1) ranked by `abs_dL`, the difference in laterality score. The HESTA score is `L_pair`, the score smoothed by the valley fit along the section order of the embryo; the fit pools neighbouring sections to a common value, so a difference of exactly 0 can come from pooling. The raw scores `L_raw` are printed as well.

`orientation = "test both"` means the side of the MOSTA section is unknown, so the pair may be mirror images. MGW compares only distances within each section and cannot tell a section from its mirror image; running both orientations would return the same coupling. The **mirror test** further down checks instead which orientation the labels and the ortholog expression support.

In [ ]:
# Stage correspondence quality (notes, Table 1)
STAGE_QUALITY = {
    ("CS12-13", "E9.5"): "good", ("CS14-15", "E10.5"): "good",
    ("CS17", "E11.5"): "ambiguous", ("CS17", "E12.5"): "ambiguous",
    ("CS18", "E12.5"): "good",
    ("CS19", "E12.5"): "ambiguous", ("CS19", "E13.5"): "ambiguous",
    ("CS20", "E13.5"): "good", ("CS23", "E14.5"): "weak",
}

pairs = pd.read_csv(LATERALITY_DIR / "laterality_pairs.csv")
scores = pd.read_csv(LATERALITY_DIR / "laterality_scores.csv", dtype={"embryo": str}).set_index("key")
fits = pd.read_csv(LATERALITY_DIR / "laterality_hesta_fits.csv", dtype={"embryo": str})

def key(dataset, sec):
    return f"HESTA:{sec}_HESTA" if dataset == "HESTA" else f"MOSTA:{sec}.MOSTA"

def name(k):
    return k.split(":", 1)[1].removesuffix("_HESTA").removesuffix(".MOSTA")

h_key = key("HESTA", HESTA_SECTION)
cand = pairs[(pairs.HESTA == h_key) & pairs.MOSTA.notna()].sort_values("abs_dL")
if cand.empty:
    raise ValueError(f"{HESTA_SECTION} has no MOSTA candidate in {LATERALITY_DIR / 'laterality_pairs.csv'}")
m_key = cand[cand.best].MOSTA.iloc[0] if MOSTA_SECTION is None else key("MOSTA", MOSTA_SECTION)
MOSTA_SEC = name(m_key)
if m_key not in scores.index:
    raise KeyError(f"{MOSTA_SEC} has no laterality score")

h, m = scores.loc[h_key], scores.loc[m_key]
pair_row = cand[cand.MOSTA == m_key]
fit = fits[(fits.stage == h.stage) & (fits.embryo == h.embryo)]
pair_info = {
    "HESTA": HESTA_SECTION, "MOSTA": MOSTA_SEC,
    "stages": f"{h.stage} / {m.stage}",
    "stage match": STAGE_QUALITY.get((h.stage, m.stage), "not in Table 1"),
    "L_pair (HESTA / MOSTA)": f"{h.L_pair:.3f} / {m.L_pair:.3f}",
    "L_raw (HESTA / MOSTA)": f"{h.L_laterality:.3f} / {m.L_laterality:.3f}",
    "abs_dL": abs(h.L_pair - m.L_pair),
    "HESTA side": h.side,
    "orientation": pair_row.orientation.iloc[0] if len(pair_row) else "n/a (manual pair)",
    "embryo fit": (f"{fit['shape'].iloc[0]}, rms {fit.fit_rms.iloc[0]:.3f}, p {fit.p_value.iloc[0]:.3f}"
                   if len(fit) else "n/a"),
}
print(pd.Series(pair_info).to_string())
print("\nMOSTA candidates for", HESTA_SECTION)
print(cand[["MOSTA", "L_HESTA", "L_MOSTA", "abs_dL", "orientation", "best"]].head(6).round(3).to_string(index=False))

# Control: a section of another HESTA embryo of the same stage, closest L_pair
CONTROL = CONTROL_SECTION
if RUN_CONTROL and CONTROL is None:
    hs = scores[(scores.dataset == "HESTA") & scores.L_pair.notna()]
    others = hs[(hs.stage == h.stage) & (hs.embryo != h.embryo)]
    if len(others):
        CONTROL = name((others.L_pair - h.L_pair).abs().idxmin())
if RUN_CONTROL and CONTROL is None:
    print(f"\nNo control: {h.stage} has no second HESTA embryo.")
elif RUN_CONTROL:
    c = scores.loc[key("HESTA", CONTROL)]
    print(f"\nControl: {CONTROL} (stage {c.stage}, embryo {c.embryo}, side {c.side}), "
          f"L_pair {c.L_pair:.3f}, |dL| to {HESTA_SECTION} = {abs(c.L_pair - h.L_pair):.3f}")

check_files(("HESTA", HESTA_SECTION), ("MOSTA", MOSTA_SEC), *([("HESTA", CONTROL)] if RUN_CONTROL and CONTROL else []))

## Shared organ labels

HESTA (`celltype`) and MOSTA (`annotation`) use different vocabularies. For the evaluation both are mapped to a coarse set of organ classes. Blood and cavity are `Excluded`; labels without a clear counterpart map to `Other`. Neither is scored. Labels missing from the map are printed when a section is loaded; add them here.

In [ ]:
EXCLUDED = "Excluded"
HARMONIZE = {
    # nervous system
    "brain": "Brain", "meninges": "Brain", "choroid plexus": "Brain", "pituitary": "Brain", "pituitary gland": "Brain",
    "spinal cord": "Spinal cord",
    "notochord": "Notochord",
    "ganglion": "Ganglion", "dorsal root ganglion": "Ganglion", "sympathetic nerve": "Ganglion",
    "facial nerve": "Ganglion", "neural crest": "Ganglion",
    "eye": "Eye", "lens": "Eye", "retina": "Eye", "optic nerve": "Eye",
    "inner ear": "Ear", "otic vesicle": "Ear",
    # organs
    "heart": "Heart",
    "liver": "Liver", "hepatic diverticulum": "Liver", "septum transversum": "Liver",
    "gut": "Gut", "gi tract": "Gut", "primitive gut": "Gut", "primitive gut tube": "Gut", "stomach": "Gut",
    "esophagus": "Gut", "pancreas": "Gut", "pancreas bud": "Gut", "pancreas primordium": "Gut",
    "lung": "Lung", "lung primordium": "Lung", "trachea": "Lung",
    "mesonephron": "Urogenital", "mesonephros": "Urogenital", "pronephron": "Urogenital", "kidney": "Urogenital",
    "metanephros": "Urogenital", "urogenital ridge": "Urogenital", "gonad": "Urogenital", "testis": "Urogenital",
    "ovary": "Urogenital", "adrenal gland": "Urogenital", "bladder": "Urogenital", "urethra": "Urogenital",
    "cloaca": "Urogenital",
    # musculoskeletal
    "somite": "Muscle", "myotome": "Muscle", "dermomyotome": "Muscle", "skeletal muscle": "Muscle", "muscle": "Muscle",
    "sclerotome": "Skeleton", "cartilage": "Skeleton", "cartilage primordium": "Skeleton", "bone": "Skeleton",
    "cranial skeleton": "Skeleton",
    "limb bud": "Limb", "limb": "Limb", "limb ectoderm": "Limb",
    # head and surface
    "branchial arch": "Craniofacial", "pharyngeal": "Craniofacial", "jaw and tooth": "Craniofacial",
    "tongue": "Craniofacial", "submandibular gland": "Craniofacial", "olfactory epithelium": "Craniofacial",
    "thyroid": "Craniofacial",
    "epidermis": "Epidermis", "surface ectoderm": "Epidermis",
    "mesenchyme": "Mesenchyme", "head mesenchyme": "Mesenchyme", "connective tissue": "Mesenchyme",
    "mesentery": "Mesenchyme", "mesothelium": "Mesenchyme", "adipose tissue": "Mesenchyme",
    # no clear counterpart
    "smooth muscle": "Other", "mucosal epithelium": "Other", "thymus": "Other", "spleen": "Other",
    # not scored
    "cavity": EXCLUDED, "extraembryonic tissue": EXCLUDED, "erythroblasts": EXCLUDED,
    "blood vessel": EXCLUDED, "endothelium": EXCLUDED, "agm": EXCLUDED,
}
NOT_SCORED = {"Other", EXCLUDED}

def harmonize(labels):
    low = pd.Series(np.asarray(labels, dtype=str)).str.lower()
    unknown = sorted(set(low) - set(HARMONIZE))
    return low.map(HARMONIZE).fillna("Other").to_numpy(dtype=str), unknown

## Helpers

**Loading.** Each section is read, the counts are moved to `X`, `DROP_LABELS` are removed, and the section is subsampled to `MAX_OBS` bins right away (normalization works per bin, so it does not matter that it runs after subsampling). The raw subsampled sections are kept, so the control can reuse the HESTA section.

**Features.** `make_features` log-normalizes copies of two sections and writes the PCA that MGW uses to `obsm["X_pca"]`: per section in `"native"` mode, joint over the shared genes in `"orthologs"` mode and for two HESTA sections. It also returns the log-normalized expression of the evaluation genes (the `N_EVAL_GENES` most variable 1:1 orthologs, without mitochondrial genes) on both sections.

In [ ]:
def load_section(dataset, sec):
    adata = sc.read_h5ad(section_path(dataset, sec))
    counts, label_col = ("counts", "celltype") if dataset == "HESTA" else ("count", "annotation")
    # HESTA X is log-normalized, MOSTA X is something else; both keep raw counts in a layer
    adata.X = adata.layers.pop(counts)
    keep = ~adata.obs[label_col].astype(str).str.lower().isin([l.lower() for l in DROP_LABELS]).to_numpy()
    adata = util.downsample_per_slice(adata[keep], MAX_OBS) if MAX_OBS else adata[keep].copy()

    # Keep only what the pipeline needs; coordinates may be stored as uint32
    out = ad.AnnData(sp.csr_matrix(adata.X, dtype=np.float32),
                     obs=pd.DataFrame(index=[f"{sec}_{n}" for n in adata.obs_names]),
                     var=pd.DataFrame(index=adata.var_names.astype(str)))
    out.obsm["spatial"] = np.asarray(adata.obsm["spatial"][:, :2], dtype=np.float64)
    out.obs["label"] = adata.obs[label_col].astype(str).to_numpy()
    out.obs["label_h"], unknown = harmonize(out.obs["label"])
    out.uns["dataset"], out.uns["section"] = dataset, sec
    del adata
    gc.collect()
    print(f"[{sec}] {out.n_obs:,} bins (of {keep.size:,}, {keep.size - keep.sum():,} dropped) x {out.n_vars:,} genes")
    if unknown:
        print(f"  labels not in HARMONIZE (scored as Other): {unknown}")
    return out


def gene_var(X):
    mean = np.asarray(X.mean(0)).ravel()
    sq = np.asarray(X.multiply(X).mean(0)).ravel() if sp.issparse(X) else (X ** 2).mean(0)
    return sq - mean ** 2


def dense(X):
    return np.asarray(X.toarray() if sp.issparse(X) else X, dtype=np.float32)


def make_features(A_raw, B_raw, mode, eval_genes=None):
    """Log-normalize copies of A and B, set obsm['X_pca'], and return the evaluation expression."""
    A, B = A_raw.copy(), B_raw.copy()
    for a in (A, B):
        sc.pp.normalize_total(a)
        sc.pp.log1p(a)
    cross = B.uns["dataset"] == "MOSTA"
    if cross:
        gh, gb = orthologs.restrict_to_orthologs(A.var_names, B.var_names, ORTHO)
    else:
        gh = gb = np.intersect1d(A.var_names, B.var_names)
    XA, XB = A[:, gh].X, B[:, gb].X   # matched columns, named by the human symbols gh

    if cross and mode == "native":
        for a in (A, B):
            sc.pp.pca(a, n_comps=PCA_COMP, random_state=0)
        how = f"per-section PCA on all genes ({A.n_vars:,} human, {B.n_vars:,} mouse)"
    else:
        joint = ad.AnnData(sp.vstack([XA, XB]).tocsr())
        sc.pp.pca(joint, n_comps=PCA_COMP, random_state=0)
        A.obsm["X_pca"], B.obsm["X_pca"] = joint.obsm["X_pca"][:A.n_obs], joint.obsm["X_pca"][A.n_obs:]
        how = f"joint PCA on {len(gh):,} shared {'1:1 orthologs' if cross else 'genes'}"
    print(f"[features] {A.uns['section']} vs {B.uns['section']}: {how}")

    if eval_genes is None:
        ok = ~pd.Index(gh).str.upper().str.startswith("MT-")
        rank = (rankdata(-gene_var(XA)) + rankdata(-gene_var(XB))) / 2
        rank[~ok] = np.inf
        eval_genes = np.asarray(gh)[np.argsort(rank)[:N_EVAL_GENES]]
    idx = pd.Index(gh).get_indexer(eval_genes)
    if (idx < 0).any():
        print(f"  {(idx < 0).sum()} evaluation genes missing in this pair; skipped")
    ev = dict(genes=np.asarray(eval_genes)[idx >= 0], Y_A=dense(XA[:, idx[idx >= 0]]), Y_B=dense(XB[:, idx[idx >= 0]]),
              n_shared=len(gh))
    return A, B, ev

**MGW and diagnostics.** `preprocess` and `align` wrap `mgw_preprocess` and `mgw_align_core` with the settings of `hesta_align.ipynb`. `align` also computes the GW energy of the final coupling (both cost matrices are scaled to a maximum of 1, so values are comparable between runs) and then frees the n × n matrices, so the control run fits in memory. The plotting and orientation functions are those of `hesta_align.ipynb`.

**Evaluation.**
- *Label transfer*: B's organ classes are moved to A through `P`. Scored are A bins whose class also occurs in B (a class missing in B cannot be predicted); `Other` or `Excluded` predictions count as wrong. Balanced accuracy is the mean recall over these classes.
- *Ortholog co-expression*: for each evaluation gene, the Pearson correlation over A's bins between its human expression and the mouse (or control) expression transported through `P`, $\hat Y = \mathrm{diag}(P\mathbf 1)^{-1} P Y_B$. The baseline transports through `P` after shuffling B's bins, which keeps the marginals but destroys the spatial correspondence.
- *Mirror test*: A is rotated, with and without a reflection, onto B (both centred and scaled to unit RMS radius), and every A bin takes its nearest B bin. The best rotation and the best reflection are scored by the same two measures.

In [ ]:
def preprocess(A, B):
    return mgw.mgw_preprocess(
        A, B, PCA_comp=PCA_COMP, CCA_comp=CCA_COMP, use_cca_feeler=True,
        use_pca_X=True, use_pca_Z=True, log1p_X=True, log1p_Z=True,
        verbose=True, feature_only=False, spatial_only=True, rep_norm="zscore")


def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def gw_energy(C1, C2, P):
    """GW objective of P in float32, on the GPU if it fits."""
    def run(dev):
        C1t, C2t, Pt = (torch.as_tensor(np.asarray(x), dtype=torch.float32, device=dev) for x in (C1, C2, P))
        p, q = Pt.sum(1), Pt.sum(0)
        return float(p @ (C1t * C1t) @ p + q @ (C2t * C2t) @ q - 2 * ((C1t @ Pt) * (Pt @ C2t)).sum())
    if device == "cuda":
        try:
            return run("cuda")
        except torch.cuda.OutOfMemoryError:
            free_memory()
    return run("cpu")


def align(pre, tag):
    out = mgw.mgw_align_core(
        pre, widths=PHI_ARC, lr=LR, niter=N_ITER, knn_k=KNN_K, geodesic_eps=GEODESIC_EPS,
        save_dir=None, tag=tag, verbose=True, plot_net=PLOT_NET, gw_params=GW_PARAMS,
        seed=SEED, deterministic=True, n_restarts=1)
    res = dict(P=np.asarray(out["P"]), xs=out["xs"], xs2=out["xs2"])
    res["gw_energy"] = gw_energy(out["C_M"], out["C_N"], res["P"])
    print(f"GW energy {res['gw_energy']:.4e}")
    del out
    free_memory()
    return res


def plot_coupling_lines(xa, xb, P, title, names, n_lines=1500, seed=0):
    """Draw B in the frame the coupling implies (reflection allowed); join random A bins to their peak B target."""
    P = np.asarray(P)
    xb_aligned, _, _, _, _ = align_vis.coupling_procrustes(xa, xb, P, ensure_rotation=False)
    idx = np.random.default_rng(seed).choice(len(xa), min(len(xa), n_lines), replace=False)
    jdx = P[idx].argmax(1)
    shift = np.array([xa[:, 0].max() - xb_aligned[:, 0].min() + 0.3 * np.ptp(xa[:, 0]), 0.0])

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.scatter(*xa.T, s=1, c="C0", label=names[0])
    ax.scatter(*(xb_aligned + shift).T, s=1, c="C1", label=f"{names[1]} (coupling frame)")
    ax.add_collection(LineCollection(np.stack([xa[idx], xb_aligned[jdx] + shift], axis=1),
                                     colors="k", alpha=0.08, lw=0.5))
    ax.set_title(f"{title}: peak target of {len(idx):,} random {names[0]} bins")
    ax.set_aspect("equal"); ax.axis("off")
    ax.legend(markerscale=8, frameon=False)
    plt.show()


def orientation_report(xa, xb, P):
    """Rigid fit of A onto its barycentric image in B, as a rotation and as a reflection. Returns det."""
    P = np.asarray(P)
    Y = (P @ xb.astype(P.dtype)) / P.sum(1, keepdims=True)
    Xc, Yc = xa - xa.mean(0), Y - Y.mean(0)
    U, _, Vt = np.linalg.svd(Xc.T @ Yc)
    d = np.sign(np.linalg.det(U @ Vt))
    print(f"orientation det = {d:+.0f}")
    for kind, Q in [("rotation", U @ np.diag([1, d]) @ Vt), ("reflection", U @ np.diag([1, -d]) @ Vt)]:
        rms = np.sqrt(((Xc @ Q - Yc) ** 2).sum(1).mean())
        angle = np.degrees(np.arctan2(Q[0, 1], Q[0, 0]))
        what = f"angle {angle:+4.0f}°" if kind == "rotation" else f"mirror axis {angle / 2:+4.0f}°"
        print(f"best {kind:<10}  {what}  rms {rms:.4f}")
    return int(d)


def row_entropy(p):
    p = p / p.sum()
    return -(p * np.log(p + 1e-30)).sum()


def feeler_report(pre, names):
    feeler = pre["feeler"]
    xa = feeler["xs"][feeler["meta"]["idxX_feeler"]]
    xb = feeler["xt"][feeler["meta"]["idxZ_feeler"]]
    P_f = np.asarray(feeler["P_feeler"])
    plot_coupling_lines(xa, xb, P_f, "CCA feeler coupling", names)
    det = orientation_report(xa, xb, P_f)
    H = np.mean([row_entropy(p) for p in P_f])
    print(f"mean row entropy {H:.2f}  (uniform {np.log(P_f.shape[1]):.2f}, one-to-one 0)")
    return dict(feeler_det=det, feeler_entropy=H)


# ---------- evaluation
def scored_classes(a_true, b_labels):
    return sorted((set(a_true) & set(b_labels)) - NOT_SCORED)


def label_transfer(P, A, B):
    a_true, b_lab = A.obs["label_h"].to_numpy(str), B.obs["label_h"].to_numpy(str)
    pred, _, _ = plotting.project_labels_via_P(P.T, b_lab, direction="A_to_B")
    classes = scored_classes(a_true, b_lab)
    mask = np.isin(a_true, classes)
    recall = {c: float((pred[a_true == c] == c).mean()) for c in classes}
    return dict(pred=np.asarray(pred, dtype=str), classes=classes, mask=mask,
                acc=float((pred[mask] == a_true[mask]).mean()), bal_acc=float(np.mean(list(recall.values()))),
                recall=recall)


def colwise_r(X, Y):
    Xc, Yc = X - X.mean(0), Y - Y.mean(0)
    den = np.sqrt((Xc ** 2).sum(0) * (Yc ** 2).sum(0))
    return np.where(den > 0, (Xc * Yc).sum(0) / np.where(den > 0, den, 1), np.nan)


def coexpression(P, ev, seed=0):
    rowsum = P.sum(1, keepdims=True)
    r = colwise_r(ev["Y_A"], (P @ ev["Y_B"]) / rowsum)
    perm = np.random.default_rng(seed).permutation(len(ev["Y_B"]))
    r0 = colwise_r(ev["Y_A"], (P @ ev["Y_B"][perm]) / rowsum)
    return r, r0


def rigid_scan(A, B, ev, step=5):
    """Nearest-neighbour maps for rotations of A onto B, with and without a reflection."""
    def unit(xy):
        xy = xy - xy.mean(0)
        return xy / np.sqrt((xy ** 2).sum(1).mean())
    a, b = unit(A.obsm["spatial"]), unit(B.obsm["spatial"])
    tree = cKDTree(b)
    a_true, b_lab = A.obs["label_h"].to_numpy(str), B.obs["label_h"].to_numpy(str)
    mask = np.isin(a_true, scored_classes(a_true, b_lab))
    rows = []
    for flip in (False, True):
        xy0 = a * [1, -1] if flip else a
        for deg in range(0, 360, step):
            t = np.radians(deg)
            nn = tree.query(xy0 @ np.array([[np.cos(t), np.sin(t)], [-np.sin(t), np.cos(t)]]))[1]
            rows.append(dict(map="reflection" if flip else "rotation", angle=deg,
                             acc=(b_lab[nn][mask] == a_true[mask]).mean(),
                             median_r=np.nanmedian(colwise_r(ev["Y_A"], ev["Y_B"][nn]))))
    return pd.DataFrame(rows)


def mirror_report(scan, lt, r):
    best = []
    for kind, g in scan.groupby("map", sort=False):
        a, e = g.loc[g.acc.idxmax()], g.loc[g.median_r.idxmax()]
        best.append(dict(map=kind, angle_by_labels=a.angle, acc=a.acc, angle_by_expr=e.angle, median_r=e.median_r))
    best = pd.DataFrame(best).set_index("map")
    print(best.round(3).to_string())
    print(f"MGW coupling: acc {lt['acc']:.3f}, median r {np.nanmedian(r):.3f}")
    by_lab = best.acc.idxmax()
    by_expr = best.median_r.idxmax()
    print(f"labels favour the {by_lab}, ortholog expression favours the {by_expr}")
    return best


def plot_label_panels(A, B, pred, names):
    a_lab, b_lab = A.obs["label_h"].to_numpy(str), B.obs["label_h"].to_numpy(str)
    _, _, classes = align_vis.coarsen_to_shared_annotations(a_lab, b_lab, max_classes=8)
    palette = dict(zip([c for c in classes if c != "Other"], align_vis.CATEGORY_COLORS))
    palette["Other"] = "#CBD5E1"
    coarse = lambda lab: np.where(np.isin(lab, list(palette)), lab, "Other")
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
    panels = [(B.obsm["spatial"], b_lab, f"{names[1]}: annotated"),
              (A.obsm["spatial"], a_lab, f"{names[0]}: annotated"),
              (A.obsm["spatial"], pred, f"{names[0]}: transferred from {names[1]}")]
    for a, (xy, lab, title) in zip(ax, panels):
        a.scatter(*xy.T, c=[palette[l] for l in coarse(lab)], s=2, linewidths=0)
        a.set_title(title); a.set_aspect("equal"); a.axis("off")
    fig.legend(handles=[Line2D([0], [0], marker="o", ls="", color=palette[c], label=c) for c in palette],
               loc="outside lower center", ncol=len(palette), frameon=False)
    plt.show()


def plot_confusion(A, lt, title):
    a_true = A.obs["label_h"].to_numpy(str)[lt["mask"]]
    pred = lt["pred"][lt["mask"]]
    cols = lt["classes"] + sorted(set(pred) - set(lt["classes"]))
    M = pd.crosstab(pd.Categorical(a_true, lt["classes"]), pd.Categorical(pred, cols),
                    normalize="index", dropna=False)
    fig, ax = plt.subplots(figsize=(0.55 * len(cols) + 2.5, 0.5 * len(lt["classes"]) + 1.5))
    im = ax.imshow(M.to_numpy(), cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(cols)), cols, rotation=60, ha="right")
    ax.set_yticks(range(len(lt["classes"])), lt["classes"])
    ax.set_xlabel("transferred class"); ax.set_ylabel("annotated class")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label="fraction of annotated class")
    plt.show()


def plot_coexpression(r, r0, title):
    fig, ax = plt.subplots(figsize=(6, 3.2))
    bins = np.linspace(-0.3, 1, 53)
    ax.hist(r0[np.isfinite(r0)], bins=bins, color="#CBD5E1", label=f"shuffled B (median {np.nanmedian(r0):.3f})")
    ax.hist(r[np.isfinite(r)], bins=bins, color="#146C94", alpha=0.8, label=f"coupling (median {np.nanmedian(r):.3f})")
    ax.set_xlabel("Pearson r over A bins"); ax.set_ylabel("genes"); ax.set_title(title)
    ax.legend(frameon=False)
    plt.show()


def summary_row(run, names, kind, lt, r, r0, mirror, feeler):
    return {
        "run": kind, "A": names[0], "B": names[1], "feature_mode": FEATURE_MODE if kind == "cross-species" else "joint",
        "n_A": len(run["xs"]), "n_B": len(run["xs2"]), "gw_energy": run["gw_energy"],
        "label_acc": lt["acc"], "label_bal_acc": lt["bal_acc"], "n_classes": len(lt["classes"]),
        "coexpr_median_r": float(np.nanmedian(r)), "coexpr_shuffled_r": float(np.nanmedian(r0)),
        "det": run["det"], **feeler,
        "rot_acc": mirror.loc["rotation", "acc"], "refl_acc": mirror.loc["reflection", "acc"],
        "rot_r": mirror.loc["rotation", "median_r"], "refl_r": mirror.loc["reflection", "median_r"],
    }

# 1. Load the sections and build the features

In [ ]:
A_raw = load_section("HESTA", HESTA_SECTION)
B_raw = load_section("MOSTA", MOSTA_SEC)
A, B, ev = make_features(A_raw, B_raw, FEATURE_MODE)
names = (HESTA_SECTION, MOSTA_SEC)
print(f"shared 1:1 orthologs: {ev['n_shared']:,}; evaluation genes: {len(ev['genes'])}")

# 2. Check the CCA feeler alignment.

The CCA is fit on pairs from a small spatial-only GW between random subsets of the two sections. If that coupling is mirrored or diffuse, the features are fit to the wrong partners. The plot draws B in the frame the feeler coupling implies (reflection allowed). The orientation check fits a rotation and a reflection to it. The row entropy measures how spread out it is.

In [ ]:
pre = preprocess(A, B)
A, B = pre["A_"], pre["B_"]
feeler_stats = feeler_report(pre, names)

# 3. MGW alignment

In [ ]:
run = align(pre, tag=f"HM_{HESTA_SECTION}_{MOSTA_SEC}_{FEATURE_MODE}_seed{SEED}")
P = run["P"]
del pre
free_memory()

# We can plot the alignment after mapping B into A's frame with a Procrustes fit to the coupling.

The fit allows a reflection (`ensure_rotation=False`), so B is drawn the way the coupling actually places it. Lines join a random subset of A bins to their peak target in B. The orientation check below fits a rotation and a reflection; `det = -1` means the coupling is a mirror image of a rotation.

In [ ]:
plot_coupling_lines(run["xs"], run["xs2"], P, "MGW coupling", names)
run["det"] = orientation_report(run["xs"], run["xs2"], P)

# We can also view the alignment as two tilted layers joined by the coupling's peak links.

In [ ]:
top, bottom, _ = align_vis.coarsen_to_shared_annotations(A.obs["label_h"].to_numpy(str),
                                                         B.obs["label_h"].to_numpy(str), max_classes=7)
_ = align_vis.plot_tilted_alignment(
    A.obsm["spatial"], B.obsm["spatial"], top, bottom, P,
    dataset_title=f"Human {HESTA_SECTION} → mouse {MOSTA_SEC} · MGW ({FEATURE_MODE})",
    top_name=HESTA_SECTION, bottom_name=MOSTA_SEC,
    label_description="shared organ classes", k_per_point=3)

# Label transfer: mouse organ classes moved onto the human section.

In [ ]:
lt = label_transfer(P, A, B)
print(f"label transfer {MOSTA_SEC} → {HESTA_SECTION}: accuracy {lt['acc']:.3f}, "
      f"balanced accuracy {lt['bal_acc']:.3f} over {len(lt['classes'])} classes")
print(pd.Series(lt["recall"]).round(3).to_string())
plot_label_panels(A, B, lt["pred"], names)
plot_confusion(A, lt, f"{MOSTA_SEC} → {HESTA_SECTION}")

# Ortholog co-expression: does the transported mouse expression match the human expression?

In [ ]:
r, r0 = coexpression(P, ev)
plot_coexpression(r, r0, f"{len(ev['genes'])} orthologs, {MOSTA_SEC} → {HESTA_SECTION}")
top_genes = pd.Series(r, index=ev["genes"]).sort_values(ascending=False)
print("best transported genes:", ", ".join(f"{g} ({v:.2f})" for g, v in top_genes.head(12).items()))

# Mirror test: compare the coupling with the best rigid maps.

This uses the annotations and the orthologs, so it is a diagnostic, not part of MGW. If the reflection beats the rotation (or vice versa) by both measures, the pair has that orientation; MGW agrees if its `det` matches (+1 rotation, −1 reflection) and its scores are close to the better rigid map. Rigid maps ignore the differences in shape between species, so their absolute scores are lower bounds.

In [ ]:
scan = rigid_scan(A, B, ev)
mirror = mirror_report(scan, lt, r)

fig, ax = plt.subplots(1, 2, figsize=(11, 3), sharex=True)
for kind, g in scan.groupby("map", sort=False):
    ax[0].plot(g.angle, g.acc, label=kind)
    ax[1].plot(g.angle, g.median_r, label=kind)
ax[0].axhline(lt["acc"], color="k", ls="--", lw=1, label="MGW")
ax[1].axhline(np.nanmedian(r), color="k", ls="--", lw=1, label="MGW")
ax[0].set_ylabel("label accuracy"); ax[1].set_ylabel("median ortholog r")
for a in ax:
    a.set_xlabel("rotation angle (°)")
ax[0].legend(frameon=False)
plt.show()

# 4. Control: the same human section against another human embryo

Same pipeline, with both sections in the human gene space (joint PCA over the shared genes). The co-expression check uses the same evaluation genes as above, so both runs are scored on the same genes.

In [ ]:
rows = [summary_row(run, names, "cross-species", lt, r, r0, mirror, feeler_stats)]
couplings = {"cross-species": (names, run, A, B)}

if RUN_CONTROL and CONTROL:
    C_raw = load_section("HESTA", CONTROL)
    A_c, C_c, ev_c = make_features(A_raw, C_raw, "joint", eval_genes=ev["genes"])
    names_c = (HESTA_SECTION, CONTROL)
    pre_c = preprocess(A_c, C_c)
    A_c, C_c = pre_c["A_"], pre_c["B_"]
    feeler_c = feeler_report(pre_c, names_c)
    run_c = align(pre_c, tag=f"HH_{HESTA_SECTION}_{CONTROL}_seed{SEED}")
    del pre_c
    free_memory()

    plot_coupling_lines(run_c["xs"], run_c["xs2"], run_c["P"], "MGW coupling (control)", names_c)
    run_c["det"] = orientation_report(run_c["xs"], run_c["xs2"], run_c["P"])
    lt_c = label_transfer(run_c["P"], A_c, C_c)
    print(f"label transfer {CONTROL} → {HESTA_SECTION}: accuracy {lt_c['acc']:.3f}, "
          f"balanced accuracy {lt_c['bal_acc']:.3f} over {len(lt_c['classes'])} classes")
    plot_label_panels(A_c, C_c, lt_c["pred"], names_c)
    r_c, r0_c = coexpression(run_c["P"], ev_c)
    plot_coexpression(r_c, r0_c, f"{len(ev_c['genes'])} genes, {CONTROL} → {HESTA_SECTION}")
    mirror_c = mirror_report(rigid_scan(A_c, C_c, ev_c), lt_c, r_c)
    rows.append(summary_row(run_c, names_c, "control (HESTA)", lt_c, r_c, r0_c, mirror_c, feeler_c))
    couplings["control (HESTA)"] = (names_c, run_c, A_c, C_c)
else:
    print("control skipped")

# Summary

One row per alignment. The rows are appended to `human_mouse_summary.csv` in the alignments folder, with the pair and the main settings, so runs with other sections and feature modes add up to one table.

In [ ]:
summary = pd.DataFrame(rows)
meta = {"date": datetime.now().isoformat(timespec="seconds"),
        "stages": pair_info["stages"], "stage_match": pair_info["stage match"],
        "abs_dL": pair_info["abs_dL"], "HESTA_side": pair_info["HESTA side"],
        "orientation": pair_info["orientation"], "max_obs": MAX_OBS, "n_iter": N_ITER,
        "drop_labels": ";".join(DROP_LABELS), "seed": SEED}
summary = summary.assign(**meta)
print(summary.drop(columns=list(meta)).set_index("run").T.to_string())
summary.to_csv(SUMMARY_CSV, mode="a", header=not SUMMARY_CSV.exists(), index=False)
print("appended to", SUMMARY_CSV)

# Save the couplings for later analysis.

`P` rows and columns refer to the subsampled bins, so their names are saved with it.

In [ ]:
for kind, ((na, nb), res, a, b) in couplings.items():
    suffix = FEATURE_MODE if kind == "cross-species" else "control"
    path = SAVE_DIR / f"P_{na}_{nb}_{suffix}.npz"
    np.savez(path, P=res["P"].astype(np.float32), obs_A=a.obs_names.to_numpy(str), obs_B=b.obs_names.to_numpy(str))
    print("Saved", path)